In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
import os

In [ ]:
folder = r"D:\2026夏季学期\传播数据分析\数据文件"
data_file = os.path.join(folder, "百度指数数据.csv")
lifecycle_file = os.path.join(folder, "热词生命周期计算.csv")

# 第1步：读取CSV文件
def read_csv_auto_encoding(file_path):
    encodings = ['utf-8-sig', 'gbk', 'gb18030']
    for enc in encodings:
        try:
            df = pd.read_csv(file_path, encoding=enc)
            print(f"用 {enc} 编码读取文件成功")
            return df
        except UnicodeDecodeError:
            continue
    # 若所有编码都失败，用utf-8忽略错误读取
    print("所有编码均失败，用utf-8忽略错误读取")
    return pd.read_csv(file_path, encoding='utf-8', on_bad_lines='skip')

In [3]:
# 读取原始热度数据
df_data = read_csv_auto_encoding(data_file)

df_data = df_data.melt(
    id_vars=['日期', '关键词'],  # 保留的主键列
    value_vars=['全部', 'PC端', '移动端'],  # 要展开的数值列
    var_name='type',  # 新的类型列名
    value_name='index'  # 新的数值列名
)

# 重命名列
df_data.rename(columns={'日期': 'date', '关键词': 'keyword'}, inplace=True)

# 数据格式转换与清洗
df_data['date'] = pd.to_datetime(df_data['date'], errors='coerce')  # 转换为日期格式
df_data['index'] = pd.to_numeric(df_data['index'], errors='coerce')  # 转换为数值格式
df_data = df_data.dropna(subset=['date', 'index'])  # 清理无效数据

# 读取生命周期起止点文件
df_life = pd.read_csv(lifecycle_file, encoding='utf-8-sig')
df_life['start_date'] = pd.to_datetime(df_life['start_date'], errors='coerce')
df_life['end_date'] = pd.to_datetime(df_life['end_date'], errors='coerce')
df_life = df_life.dropna(subset=['start_date', 'end_date'])  # 清理无效生命周期数据

用 gbk 编码读取文件成功


In [ ]:
# 第2步：对每个热词计算指标
results = []

for idx, row in df_life.iterrows():
    kw = row['keyword']
    start = row['start_date']
    end = row['end_date']
    # 筛选该热词生命周期内的数据（包含起止日）
    mask = (df_data['keyword'] == kw) & (df_data['date'] >= start) & (df_data['date'] <= end)
    kw_data = df_data.loc[mask].copy()

    if kw_data.empty:
        print(f"警告：{kw} 生命周期内无数据，跳过")
        continue

    kw_data = kw_data.sort_values('date')
    hot = kw_data['index'].values
    # 基本统计
    peak = np.max(hot)
    mean = np.mean(hot)
    life_days = (end - start).days + 1

    # 爆发速度
    peak_idx = np.argmax(hot)
    start_hot = hot[0]
    days_to_peak = peak_idx
    if days_to_peak == 0:
        rise_speed = 0
    else:
        rise_speed = (peak - start_hot) / days_to_peak

    # 偏度和峰度（使用 scipy）
    skewness_val = skew(hot)
    kurtosis_val = kurtosis(hot, fisher=True)

    # 半衰期计算
    total_hot = np.sum(hot)
    cumsum = np.cumsum(hot)
    half_index = np.searchsorted(cumsum, total_hot / 2, side='left')
    # 历时半衰期（起点到half_index的天数）
    rise_half = half_index
    # 共时半衰期（half_index到结束的天数）
    fall_half = len(hot) - half_index - 1 if half_index < len(hot) else 0
    # 半衰期趋势比（历时/共时）
    if fall_half == 0:
        trend_ratio = np.nan
    else:
        trend_ratio = rise_half / fall_half
    
    # 存储结果
    results.append({
        'keyword': kw,
        'peak': peak,
        'mean': round(mean, 2),
        'lifecycle_days': life_days,
        'rise_speed': round(rise_speed, 2),
        'skewness': round(skewness_val, 4),
        'kurtosis': round(kurtosis_val, 4),
        'rise_half_life': rise_half,
        'fall_half_life': fall_half,
        'trend_ratio': round(trend_ratio, 4) if not np.isnan(trend_ratio) else None
    })
    print(f"已完成 {kw}")

已完成 厉害了我的哥
已完成 友谊的小船
已完成 吃瓜群众
已完成 洪荒之力
已完成 老司机
已完成 葛优躺
已完成 蓝瘦香菇
已完成 辣眼睛
已完成 你有freestyle吗
已完成 你的良心不会痛吗
已完成 尬聊
已完成 怼
已完成 扎心了老铁
已完成 打call
已完成 油腻
已完成 皮皮虾我们走
已完成 还有这种操作
已完成 c位
已完成 skr
已完成 佛系
已完成 土味情话
已完成 官宣
已完成 杠精
已完成 燃烧我的卡路里
已完成 确认过眼神
已完成 锦鲤
已完成 996
已完成 不忘初心
已完成 好嗨哟
已完成 断舍离
已完成 柠檬精
已完成 硬核
已完成 道路千万条安全第一条
已完成 雨女无瓜
已完成 不约而同
已完成 云监工
已完成 光盘行动
已完成 夺冠
已完成 奥利给
已完成 好家伙
已完成 秋天的第一杯奶茶
已完成 逆行者
已完成 集美
已完成 yyds
已完成 元宇宙
已完成 双减
已完成 强国有我
已完成 破防
已完成 绝绝子
已完成 觉醒年代
已完成 躺平
已完成 一起向未来
已完成 中国式现代化
已完成 俄乌冲突
已完成 党的二十大
已完成 全过程人民民主
已完成 我的眼睛就是尺
已完成 数字经济
已完成 多巴胺穿搭
已完成 显眼包
已完成 村ba
已完成 特种兵式旅游
已完成 city不city
已完成 主理人
已完成 争气
已完成 人工智能
已完成 新质生产力
已完成 浓人淡人
已完成 deepseek
已完成 情绪价值
已完成 来财
已完成 浪浪山小妖怪
已完成 爱你老己


In [ ]:
# 第3步：保存结果
result_df = pd.DataFrame(results)
output_file = os.path.join(folder, "features2.csv")
result_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n全部处理完成！")
print(f"指标计算结果已保存至：{output_file}")
print(f"共成功处理了{len(results)}个热词。")


全部处理完成！


NameError: name 'output_file' is not defined